# Introduction
The goal of this notebook is to explore the capabilities of DQN agent on various environments and learn about the impact of the important hyper parameters of the algorithm.

This notebook is using TF-agents framework and relies on the collab notebooks of the framework. 


In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
try:
    %%tensorflow_version 2.x
except:
    pass

import os
import sys
print('inserting the following to path',os.path.abspath('../..'))
sys.path.insert(0,os.path.abspath('../..'))
print(sys.path)

inserting the following to path /home/gkoren2/PycharmProjects/remote/MLA/RL/agents
['/home/gkoren2/PycharmProjects/remote/MLA/RL/agents', '/home/gkoren2/PycharmProjects/remote/MLA/RL/agents/tf_agents/my_colabs', '/opt/anaconda3/envs/rl20/lib/python37.zip', '/opt/anaconda3/envs/rl20/lib/python3.7', '/opt/anaconda3/envs/rl20/lib/python3.7/lib-dynload', '', '/opt/anaconda3/envs/rl20/lib/python3.7/site-packages', '/opt/anaconda3/envs/rl20/lib/python3.7/site-packages/IPython/extensions', '/home/gkoren2/.ipython']


In [3]:
from __future__ import absolute_import, division, print_function

import base64
# import imageio
import IPython
import matplotlib
import matplotlib.pyplot as plt
import PIL.Image
# import pyvirtualdisplay
import json 

import tensorflow as tf

from tf_agents.agents.dqn import dqn_agent
from tf_agents.drivers import dynamic_step_driver
from tf_agents.environments import suite_gym
from tf_agents.environments import tf_py_environment
from tf_agents.eval import metric_utils
from tf_agents.metrics import tf_metrics
from tf_agents.networks import q_network
from tf_agents.policies import random_tf_policy
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.trajectories import trajectory
from tf_agents.utils import common


# my import to control gpu usage
from tf_agents.utils.my_utils import set_gpu_device


tf.compat.v1.enable_v2_behavior()
set_gpu_device('0')

1 Physical GPUs, 1 Logical GPU


In [ ]:
print(tf.version.VERSION)

# Experiments
Each experiment will include the following parameters:
 - Environment 
 - Network parameters 
 - Agent parameters 
 - Replay buffer and sampler 
 - Metrics and evaluation 
 
 
Then we'll define the training loop.
maybe a good idea is to define a backend file that will get all these parameters and will run the experiment, dumping to tensorboard and we'll read the tensorboard to the jupyter. 

it looks like the backend file can be based on train_eval.py and be run from this notebook by :
`%run -i dqn_hands_on.py`

In [ ]:
# todo: save params to params_file as json
# % run -i dqn_hands_on_be.py $params_file 

## Hyper Parameters
Lets set the default hyper parameters and in subsequent cells just change those that we want to play with.

In [4]:
from dqn_hands_on_be import DEFAULT_ROOT_DIR
from dqn_hands_on_be import def_exp_params

In [5]:
print(DEFAULT_ROOT_DIR)
print(json.dumps(def_exp_params,indent=2))

/home/gkoren2/share/Data/MLA/agents
{
  "env_name": "CartPole-v0",
  "num_iterations": 100000,
  "train_sequence_length": 1,
  "fc_layer_params": [
    100
  ],
  "input_fc_layer_params": [
    50
  ],
  "lstm_size": [
    20
  ],
  "output_fc_layer_params": [
    20
  ],
  "initial_collect_steps": 1000,
  "collect_steps_per_iteration": 1,
  "epsilon_greedy": 0.1,
  "replay_buffer_capacity": 100000,
  "target_update_tau": 0.05,
  "target_update_period": 5,
  "train_steps_per_iteration": 1,
  "batch_size": 64,
  "learning_rate": 0.001,
  "n_step_update": 1,
  "gamma": 0.99,
  "reward_scale_factor": 1.0,
  "gradient_clipping": null,
  "use_tf_functions": true,
  "num_eval_episodes": 10,
  "eval_interval": 1000,
  "train_checkpoint_interval": 10000,
  "policy_checkpoint_interval": 5000,
  "rb_checkpoint_interval": 20000,
  "log_interval": 1000,
  "summary_interval": 1000,
  "summaries_flush_secs": 10,
  "debug_summaries": false,
  "summarize_grads_and_vars": false,
  "eval_metrics_callbac

### Default Experiment Params

From this point on, the flow will be:
 - change subset of the parameters 
 - save as a json file
 - run the script with the json file as input :  
 `%run dqn_hands_on_be.py --alsologtostderr [--root_dir root_dir] [--exp_json json_file] [--n_experiments n] [--gpu 1]`

The idea is that I can run the script also from command line and not only through jupyter. that's why its better to read a json file as parameter. The challenge is when the valu eof a certain key is a function. in this case, I'll put a string in the json and another dictionary in the python file.

**TODO** : find a way to open tensorboard from jupyter to follow up on the training process.


In [ ]:
exp_params=def_exp_params
# change what you want

#set the name of the json
json_file = os.path.join(DEFAULT_ROOT_DIR,'jsons','exp1.json')
print(json_file)
# dump to file
with open(json_file, 'w') as output_file:
        json.dump(exp_params, output_file,indent=2)

In [6]:
json_file = os.path.join('/home/gkoren2/share/Data/MLA/agents','jsons','exp1.json')
%run dqn_hands_on_be.py --alsologtostderr --exp_json $json_file --gpu 0

I0115 10:04:41.849738 140547229972288 dqn_hands_on_be.py:412] loading configuration from /home/gkoren2/share/Data/MLA/agents/jsons/exp1.json
I0115 10:04:41.851556 140547229972288 dqn_hands_on_be.py:340] set seed to 1 and creating folder for results


1 Physical GPUs, 1 Logical GPU


I0115 10:04:42.652472 140547229972288 common.py:926] No checkpoint available at /home/gkoren2/share/Data/MLA/agents/results/exp1_15-01-2020_10-04-41/1/train
I0115 10:04:42.655839 140547229972288 common.py:926] No checkpoint available at /home/gkoren2/share/Data/MLA/agents/results/exp1_15-01-2020_10-04-41/1/train/policy
I0115 10:04:42.658742 140547229972288 common.py:926] No checkpoint available at /home/gkoren2/share/Data/MLA/agents/results/exp1_15-01-2020_10-04-41/1/train/replay_buffer
I0115 10:04:42.661570 140547229972288 dqn_hands_on_be.py:238] Initializing replay buffer by collecting experience for 1000 steps with a random policy.
I0115 10:04:53.429281 140547229972288 metric_utils.py:47]  
		 AverageReturn = 8.899999618530273
		 AverageEpisodeLength = 8.899999618530273
I0115 10:05:13.367955 140547229972288 dqn_hands_on_be.py:290] step = 1000, loss = 1.167294
I0115 10:05:13.370264 140547229972288 dqn_hands_on_be.py:292] 106.110 steps/sec


KeyboardInterrupt: 

# Experiments
in the following subsections we'll run the on set of atari games (the ram version, in order to save complications)